# The Singular Value Decomposition

Every matrix, of any shape, can be written $A = U \Sigma V^\top$: a rotation, a stretch along the axes and another rotation. This notebook reads the rank and the condition number from the singular values, and compresses a matrix by keeping only the largest ones.

In [ ]:
Pollis.packages("Plots")

In [ ]:
using LinearAlgebra, Plots

In [ ]:
A = [3.0 2 2; 2 3 -2]
S = svd(A)
(singular_values = S.S, check = norm(S.U * Diagonal(S.S) * S.Vt - A))

## What a matrix does to a circle

A $2 \times 2$ matrix maps the unit circle to an ellipse whose semi-axes are the singular values, along the columns of $U$.

In [ ]:
B = [2.0 1; 0.5 1.5]
Sb = svd(B)
th = range(0, 2pi, length=200)
circle = [cos.(th) sin.(th)]
ellipse = circle * transpose(B)
p = plot(circle[:, 1], circle[:, 2], aspect_ratio=:equal, label="unit circle", lw=1.5)
plot!(p, ellipse[:, 1], ellipse[:, 2], label="image under B", lw=2)
for i in 1:2
    a = Sb.S[i] * Sb.U[:, i]
    plot!(p, [0, a[1]], [0, a[2]], arrow=true, lw=2, label="sigma_$i = $(round(Sb.S[i], digits=3))")
end
p

## Rank and conditioning

The number of nonzero singular values is the rank; the ratio of the largest to the smallest is the condition number. A matrix whose columns are nearly dependent has a tiny last singular value.

In [ ]:
N = [1.0 1; 1 1.0001]
(singular_values = svdvals(N), condition_number = cond(N), rank = rank(N))

## The best low-rank approximation

Keeping the first $k$ terms $\sum_{i \le k} \sigma_i u_i v_i^\top$ gives the closest rank-$k$ matrix, with 2-norm error exactly $\sigma_{k+1}$ (Eckart and Young). Build a $60 \times 40$ matrix that is rank 3 plus a little noise, and see the three large singular values stand out.

In [ ]:
using Random
rng = Xoshiro(7)
low = randn(rng, 60, 3) * randn(rng, 3, 40)
Mx = low + 0.05 * randn(rng, 60, 40)
s = svdvals(Mx)
bar(s, label="singular values", xlabel="index", legend=:topright)

In [ ]:
Sm = svd(Mx)
approx(k) = Sm.U[:, 1:k] * Diagonal(Sm.S[1:k]) * Sm.Vt[1:k, :]
[(rank = k, error_2norm = opnorm(Mx - approx(k)), next_singular_value = Sm.S[k + 1], share_of_energy = sum(Sm.S[1:k] .^ 2) / sum(Sm.S .^ 2)) for k in 1:5]

Three components carry nearly all the energy: the rest is the noise. This is the idea behind principal components and data compression.

## References

[Eckart and Young (1936)](https://doi.org/10.1007/BF02288367), The approximation of one matrix by another of lower rank, *Psychometrika* 1(3), 211–218.

[Golub and Kahan (1965)](https://doi.org/10.1137/0702016), Calculating the singular values and pseudo-inverse of a matrix, *SIAM Journal on Numerical Analysis, Series B* 2(2), 205–224.